# Maia Corpus v9 — Code Acquisition v1.1
Common Corpus extension. CPU Colab. Run all cells, single writer only. Original v1.0 is read-only. Metadata-first Parquet scanning avoids downloading unrelated text columns where HTTP range works. Candidates only; retain attribution and review upstream obligations before training or redistribution.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json,gzip,hashlib,os,re,time,collections,random,sqlite3
import pyarrow.parquet as pq
import fsspec,aiohttp
from huggingface_hub import HfApi,hf_hub_url
BASE=Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_4_0/maia-corpus-v9')
OLD=BASE/'code_v10'; OUT=BASE/'code_v11'; OUT.mkdir(exist_ok=True)
BATCHES=OUT/'batches'; BATCHES.mkdir(exist_ok=True)
TARGET={'Python':360_000_000,'JavaScript':240_000_000,'C':180_000_000,'C++':180_000_000,'Java':120_000_000,'TypeScript':120_000_000}
ALLOW={'mit','apache-2.0','bsd-2-clause','bsd-3-clause','isc','0bsd','zlib','unlicense','cc0-1.0'}
REPO='PleIAs/common_corpus'
MAX_FILES=1000 # per invocation; rerun to continue
EXT={'.js':'JavaScript','.jsx':'JavaScript','.ts':'TypeScript','.tsx':'TypeScript','.c':'C','.cpp':'C++','.cc':'C++','.cxx':'C++','.java':'Java'}
SECRET=re.compile(r'-----BEGIN (?:RSA |EC |OPENSSH |DSA )?PRIVATE KEY-----|\b(?:ghp_|github_pat_|sk_live_)[A-Za-z0-9_]{16,}|\bAKIA[A-Z0-9]{16}\b')
SPDX=re.compile(r'SPDX-License-Identifier:\s*([^\r\n*]+)',re.I)
def atomic(p,obj):
 t=p.with_suffix(p.suffix+'.part'); t.write_text(json.dumps(obj,ensure_ascii=False,indent=2)); os.replace(t,p)
def digest(t):return hashlib.sha256(t.encode()).hexdigest()

In [ ]:
# Rebuild index from durable batches, never trust a stale in-memory cursor.
seen=set(); totals=collections.Counter(); counts=collections.Counter(); done=set(); rejected=collections.Counter()
oldfiles=sorted((OLD/'batches').glob('*.json.gz')); assert oldfiles,'v1.0 batches missing'
def recover(p,old):
 with gzip.open(p,'rt',encoding='utf8') as f:o=json.load(f)
 for r in o['records']:
  h=digest(r['text']); assert h==r['sha256'],p
  assert h not in seen, f'Duplicate in committed corpus: {p}'
  seen.add(h); totals[r['language']]+=r['proxy_tokens']; counts[r['language']]+=1
 if not old:
  done.add((o['key'],o['start'])); rejected.update(o['rejects'])
for p in oldfiles:recover(p,True)
assert totals['Python']>=TARGET['Python'], 'Previous acquisition incomplete'
for p in sorted(BATCHES.glob('*.json.gz')):recover(p,False)
print('Recovered',dict(totals),dict(counts),flush=True)

In [ ]:
# Pin source revision and deterministic file ordering.
planpath=OUT/'plan.json'; config={'target':TARGET,'allow':sorted(ALLOW),'ratio':3,'repo':REPO,'policy':'metadata-first-v1'}
if planpath.exists():
 plan=json.loads(planpath.read_text()); assert plan['config']==config
else:
 info=HfApi().dataset_info(REPO,files_metadata=True,timeout=60)
 files=[x.rfilename for x in info.siblings if x.rfilename.endswith('.parquet')]; assert files
 random.Random(2026).shuffle(files)
 plan={'config':config,'revision':info.sha,'files':files}; atomic(planpath,plan)
print('Pinned',plan['revision'],'files',len(plan['files']))

In [ ]:
# Read metadata only, then fetch text for qualifying row groups.
def full():return all(totals[k]>=v for k,v in TARGET.items())
def licenses(x):
 ls=[x] if isinstance(x,str) else x if isinstance(x,list) else []
 return ls if ls and all(isinstance(a,str) and a.strip().lower() in ALLOW for a in ls) else []
def classify(r):
 if str(r.get('open_type','')).lower().replace('_','').replace(' ','')!='opensource':return None,'not_opensource'
 l=licenses(r.get('license'))
 if not l:return None,'license_unknown_or_excluded'
 ident=r.get('identifier'); title=r.get('title')
 if not isinstance(ident,str) or not ident.strip():return None,'missing_provenance'
 ext=Path(str(title or '').split('?')[0]).suffix.lower(); ext2=Path(ident.split('?')[0]).suffix.lower()
 lang=EXT.get(ext) or EXT.get(ext2)
 if EXT.get(ext) and EXT.get(ext2) and EXT[ext]!=EXT[ext2]:return None,'language_conflict'
 if not lang or totals[lang]>=TARGET[lang]:return None,'language_or_quota'
 return (lang,l),None
assert classify({'open_type':'OpenSource','license':'MIT','identifier':'https://github.com/x/y/blob/main/a.c','title':'a.c'})[0][0]=='C' or totals['C']>=TARGET['C']
def summary():
 report={'status':'TARGET_MET' if full() else 'PARTIAL_TARGET','proxy_tokens':dict(totals),'target':TARGET,'deficits':{k:max(0,v-totals[k]) for k,v in TARGET.items()},'documents_by_language':dict(counts),'extension_rejections':dict(rejected),'rights_status':'PERMISSIVE_SOURCE_CANDIDATES_ONLY','requires_final_tokenizer':True}
 atomic(OUT/'summary.json',report); return report
finished_path=OUT/'finished.json'; finished=set(json.loads(finished_path.read_text())) if finished_path.exists() else set()
errors=[]; attempted=0
for file in plan['files']:
 if full() or attempted>=MAX_FILES:break
 if file in finished:continue
 attempted+=1; print('READ',file,flush=True)
 try:
  url=hf_hub_url(REPO,file,repo_type='dataset',revision=plan['revision'])
  with fsspec.open(url,'rb',block_size=2**20,client_kwargs={'timeout':aiohttp.ClientTimeout(total=300,sock_read=90)}) as f:
   pf=pq.ParquetFile(f); cols=set(pf.schema_arrow.names)
   required={'open_type','license','identifier','title','text'}
   if not required<=cols:raise ValueError('Missing columns: '+str(required-cols))
   meta=sorted((cols & {'open_type','license','identifier','title','creator','collection','language','language_type','curator'}) )
   key=REPO+'@'+plan['revision']+'/'+file
   for g in range(pf.num_row_groups):
    if full():break
    if (key,g) in done:continue
    rows=pf.read_row_group(g,columns=meta).to_pylist(); candidates=[]; rej=collections.Counter(); records=[]
    for i,r in enumerate(rows):
     a,why=classify(r)
     if a:candidates.append(i)
     else:rej[why]+=1
    if candidates:
     texts=pf.read_row_group(g,columns=['text']).column('text')
     for i in candidates:
      r=rows[i]; a,why=classify(r)
      if not a:rej[why]+=1;continue
      lang,lic=a; t=texts[i].as_py()
      if not isinstance(t,str) or not 100<=len(t)<=500000 or '\x00' in t:rej['size_or_binary']+=1;continue
      if any(len(line)>1500 for line in t.splitlines()):rej['long_line']+=1;continue
      if SECRET.search(t):rej['possible_secret']+=1;continue
      if any(x.strip().lower() not in ALLOW for x in SPDX.findall(t)):rej['conflicting_spdx']+=1;continue
      h=digest(t)
      if h in seen:rej['duplicate']+=1;continue
      rec={'text':t,'sha256':h,'language':lang,'license_evidence':lic,'rights_status':'PERMISSIVE_SOURCE_CANDIDATE','repository':r['identifier'],'path':r['title'],'source':REPO,'revision':plan['revision'],'source_file':key,'row_group':g,'row':i,'characters':len(t),'proxy_tokens':round(len(t)/3),'source_metadata':r}
      records.append(rec);seen.add(h);totals[lang]+=rec['proxy_tokens'];counts[lang]+=1
    p=BATCHES/f'{digest(key)[:24]}_{g:012d}.json.gz'; tmp=p.with_suffix('.part')
    with gzip.open(tmp,'wt',encoding='utf8',compresslevel=3) as out:json.dump({'key':key,'start':g,'end':g+1,'records':records,'rejects':dict(rej)},out,ensure_ascii=False)
    os.replace(tmp,p); done.add((key,g)); rejected.update(rej)
    print('COMMIT',g,'accepted',len(records),'totals',dict(totals),flush=True);summary()
  finished.add(file);atomic(finished_path,sorted(finished))
 except Exception as e:
  errors.append({'file':file,'error':str(e)[:500]});atomic(OUT/'errors.json',errors)
  # Stop: in-memory counters may include uncommitted records. Run all cells to recover.
  raise RuntimeError('Source interrupted. Run all cells again; durable batches preserved.') from e
print(json.dumps(summary(),indent=2))

## Notes
Common Corpus documents its row-level license, identifier, title and open_type fields. Actual schema, code formatting, license values, and range-read throughput must be verified at runtime. No guarantee of reaching quotas. File extension is a heuristic, not a parser. Row groups may be large; no full corpus download is intended, but a server may transfer more bytes than projected. No automatic legal clearance. Attribution and upstream notices must be retained. If repeated schema failures occur, inspect errors.json rather than repeatedly retrying. Source: https://huggingface.co/datasets/PleIAs/common_corpus